# Residuals and Loss

In [ ]:
import numpy as np

from course_utils.loss_function import mse
from course_utils.plots import plot_scatter_plot

## Observations and Model Creation

If we want to build a model that best predicts unknown values we need methods for measuring it's performance against results we know to be true. 

First let's plot a simple data set of heights vs weights. 

In [ ]:
heights_cm = np.array([
    172, 158, 184, 166, 177,
    191, 163, 180, 155, 174,
    188, 169, 182, 160, 176,
    193, 167, 185, 171, 179
], dtype=float)

weights_kg = np.array([
    74, 61, 92, 68, 83,
    106, 57, 78, 52, 89,
    84, 98, 76, 64, 71,
    112, 73, 87, 69, 95
], dtype=float)

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)"
)

For the purpose of this exercise I'm going to guess the equation for predicting someone's Weight in kg from their Height in cm. This will be our model for this exercice.

Let's go with:

$$weight = 0.9 \times height - 80$$

We can formalise this as a python function below:

In [ ]:
def guess_weight(h: float) -> float:
    return 0.9 * h - 80

and then our function to generate an array of weights from an array of heights. 

In [ ]:
# Heights covering the full plotted range
line_heights = np.linspace(
    heights_cm.min(),
    heights_cm.max(),
    100
)

# Weights predicted across our full plotted range of heights
line_weights = guess_weight(line_heights)

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)", 
    line_values=(line_heights, line_weights)
)

## Model Evaluation

### Let’s determine whether this model was a good guess

We can begin assessing how closely our model fits the observed data by calculating the **residual** for each observation.

A residual is the vertical difference between an observed value, represented by a point, and the value predicted by our model, represented by the line.

$$\text{residual} = \text{observed value} - \text{predicted value}$$

In [ ]:
predicted_weights = guess_weight(heights_cm)
residuals = weights_kg - predicted_weights

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)", 
    line_values=(line_heights, line_weights),
    residuals=residuals
)

### So now we can just sum the residuals, right?

Intuitively your next step may be to sum the residuals and use that as a measure of how 'wrong' our guess was. Let's look and what would happen if we did that.

First let's sum the residuals:

In [ ]:
sum_of_residuals = np.sum(residuals)

print(f"Sum of residuals: {sum_of_residuals:.2f}")

### Wrong! ⚠️

At this point we could say something like *"In total our model was 48kgs away from being correct, I will now tweak my model to bring it closer to 0kgs or error"*.

However, you will see below that when we tweak our guess in order to do this, a major flaw in our measure (sum of the residuals) becomes apparent. 

Let's start by taking a second guess and seeing how that compares to our original model:

$$ \operatorname{weight} = 0.1 \times \operatorname{height} + 60 $$

In [ ]:
def guess_weight_v2(h: float) -> float:
    return 0.1 * h + 60

# Heights covering the full plotted range
line_heights_v2 = np.linspace(
    heights_cm.min(),
    heights_cm.max(),
    100
)

line_weights_v2 = guess_weight_v2(line_heights_v2)

predicted_weights_v2 = guess_weight_v2(heights_cm)
residuals_v2 = weights_kg - predicted_weights_v2

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)", 
    line_values=(line_heights_v2, line_weights_v2),
    residuals=residuals_v2
)

In [ ]:
sum_of_residuals_v2 = np.sum(residuals_v2)

print(f"Sum of residuals: {sum_of_residuals:.2f}")
print(f"Sum of residuals v2: {sum_of_residuals_v2:.2f}")
print(f"Improvement in sum of residuals: {sum_of_residuals - sum_of_residuals_v2:.2f}")

You can see clearly from the plots that the second guess is a less useful predictive model than the first.

Despite this, the sum of residuals suggests an apparent 8 kg improvement in the model’s ability to predict weight.

This happens because the sum of residuals retains their signs. Positive and negative residuals can therefore cancel each other out. A model with several large positive and negative residuals may consequently produce a sum close to zero, even though its predictions are far from the observed values.

The sum of residuals is therefore a misleading measure of predictive accuracy and is not suitable for our purpose.

## Loss: A More Reliable Measure of Model Performance

In practice, we define a **loss function** that assigns a numerical penalty to a model’s predictions. During training, we aim to find the model parameters that minimise this loss.

A loss function provides a more useful measure of a model’s predictive error than simply calculating the sum of its residuals.

Many different loss functions exist, and the choice of loss function affects how different kinds of prediction error are penalised. In this notebook, we will implement and use **mean squared error (MSE)**, which is commonly used with regression models such as the one we have guessed above.

The formula for mean squared error is:

$$
\operatorname{MSE}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(y_i-\hat{y}_i\right)^2
$$

where (n) is the number of observations, (y_i) is the observed value for observation (i)—an actual weight in our example—and (\hat{y}_i) is the corresponding value predicted by the model.

Recall that the residual for observation (i) is:

$$
\operatorname{residual}_i = y_i-\hat{y}_i
$$

MSE is therefore the average of the squared residuals.

Squaring each residual before calculating the average makes every resulting value non-negative. This prevents positive and negative residuals from cancelling each other out, solving the problem we encountered when using the sum of residuals.

Squaring also gives greater weight to larger residuals. For example, a residual of (10) contributes (100) to the calculation, whereas a residual of (2) contributes only (4). MSE therefore penalises large prediction differences more heavily than small ones.


### Puting Mean Squared Error to the Test

Now we know what **MSE** is and why it's important let's implenent this as a Python function and see how it handles our guesses from the previous section.

In [ ]:
def mse(y_observed: np.array, y_predicted: np.array) -> float:
    return np.mean((y_observed - y_predicted) ** 2)

mse_1 = mse(weights_kg, predicted_weights)
mse_2 = mse(weights_kg, predicted_weights_v2)

print(f"Mean Squared Error (Guess 1): {mse_1:.2f}")
print(f"Mean Squared Error (Guess 2): {mse_2:.2f}")

You can see above that the mean squared error gives values for how our guesses performed that are much more inline with what we see in the charts above.